# 🌳 Árbol de expresión modular: generación, construcción, recorridos y graficación

**Ejemplo base:** `sqrt(2*(x^2)+5*x-3)`, es decir, $\sqrt{2x^2 + 5x - 3}$

El programa trabaja de dos maneras:
1. **Ecuaciones escritas por ti** (como la de la imagen).
2. **Ecuaciones generadas automáticamente** a partir de una **biblioteca** de elementos permitidos, una **lista de valores válidos** y una **altura máxima** que limita el tamaño del árbol.

En ambos casos se obtiene lo mismo: validación, tabla de Shunting-Yard, árbol de expresión, recorridos preorden, inorden y postorden, dibujo del árbol, gráfica de la función y evaluación.

El proyecto está organizado como un **paquete de Python** llamado `arbol_expresion`, con un módulo por responsabilidad:

| Módulo | Responsabilidad |
|---|---|
| `config.py` | Funciones, constantes, operadores (precedencia y asociatividad) y colores |
| `errores.py` | Excepción `ErrorExpresion` y mensajes que señalan la posición del error |
| `lexico.py` | Normalización y tokenización (análisis léxico) |
| `sintaxis.py` | Menos unario, multiplicación implícita y validación sintáctica |
| `postfija.py` | Algoritmo Shunting-Yard: infija → postfija |
| `arbol.py` | Clase `Nodo`, construcción del árbol, recorridos y estadísticas |
| `evaluador.py` | Evaluación numérica con validación de dominio |
| `graficos.py` | Dibujo del árbol y gráfica de la función |
| `analizador.py` | Clase `ArbolExpresion` y función `analizar()` (une todo) |
| `generador.py` | Clases `Biblioteca` y `GeneradorExpresiones`: valores válidos, altura máxima, generación aleatoria |
| `interactivo.py` | Modo interactivo con `input()` |
| `pruebas.py` | Pruebas automáticas |
| `__init__.py` | Expone la interfaz pública del paquete |

**Flujo de datos:**

`biblioteca + altura máxima → generador → texto` (o `texto escrito por ti`)

`texto → lexico → tokens → sintaxis → tokens válidos → postfija → árbol → evaluador / graficos`

**Cómo usarlo:** `Entorno de ejecución → Ejecutar todas` (o `Ctrl + F9`).
Las celdas `%%writefile` crean cada archivo del paquete; después se importa y se usa.

## Paso 1 · Crear la carpeta del paquete

In [ ]:
import os
os.makedirs('arbol_expresion', exist_ok=True)
print("📁 Carpeta 'arbol_expresion' lista.")

## Paso 2 · Módulo `config.py`

Incluye `ALTURA_MAXIMA_LIMITE = 8`: el generador no acepta alturas mayores, porque un árbol de altura 8 ya puede tener 255 nodos.

Cada operador tiene **precedencia**, **asociatividad** y **aridad** (número de operandos):

| Operador | Significado | Precedencia | Asociatividad | Aridad |
|---|---|---|---|---|
| `+` `-` | suma, resta | 1 | izquierda | 2 |
| `*` `/` | multiplicación, división | 2 | izquierda | 2 |
| `neg` | negativo unario (`-x`) | 3 | derecha | 1 |
| `^` | potencia | 4 | derecha | 2 |

> `2^3^2 = 2^(3^2) = 512` (asociatividad derecha) y `-x^2 = -(x^2)` (el negativo tiene menor precedencia que `^`).
>
> **Extensible:** para agregar una función nueva basta con añadirla al diccionario `FUNCIONES`.

In [ ]:
%%writefile arbol_expresion/config.py
"""
config.py — Configuración del lenguaje de expresiones.

Para agregar una función nueva (por ejemplo, 'asin') basta con añadirla
al diccionario FUNCIONES; el resto de los módulos la reconocen solos.
"""
import math

# Funciones de un argumento: nombre -> implementación
FUNCIONES = {
    'sqrt': math.sqrt,   # raíz cuadrada
    'sin':  math.sin,    # seno (radianes)
    'cos':  math.cos,    # coseno (radianes)
    'tan':  math.tan,    # tangente (radianes)
    'log':  math.log10,  # logaritmo base 10
    'ln':   math.log,    # logaritmo natural
    'exp':  math.exp,    # e^x
    'abs':  abs,         # valor absoluto
}

# Constantes: nombre -> valor
CONSTANTES = {'pi': math.pi, 'e': math.e}

# Operadores: símbolo -> (precedencia, asociatividad, aridad)
OPERADORES = {
    '+':   (1, 'izq', 2),
    '-':   (1, 'izq', 2),
    '*':   (2, 'izq', 2),
    '/':   (2, 'izq', 2),
    'neg': (3, 'der', 1),   # menos unario: -x
    '^':   (4, 'der', 2),
}

# Altura máxima que acepta el generador de ecuaciones.
# Un árbol de altura 8 puede tener hasta 2^8 - 1 = 255 nodos; más que eso ya no se puede dibujar legible.
ALTURA_MAXIMA_LIMITE = 8

# Tipos de token que son operandos (hojas del árbol)
TIPOS_OPERANDO = ('NUM', 'VAR', 'CONST')

# Colores del dibujo: tipo de nodo -> (color, etiqueta de la leyenda)
COLORES = {
    'OP':    ('#4C72B0', 'Operador binario'),
    'OP_UN': ('#8172B3', 'Operador unario (neg)'),
    'FUNC':  ('#DD8452', 'Función'),
    'NUM':   ('#55A868', 'Número'),
    'VAR':   ('#C44E52', 'Variable'),
    'CONST': ('#2A9D8F', 'Constante'),
}

## Paso 3 · Módulo `errores.py`

Define la excepción propia del proyecto y la función que dibuja una flecha `^` bajo la posición exacta del error.

In [ ]:
%%writefile arbol_expresion/errores.py
"""
errores.py — Excepción propia y utilidades para mensajes de error.
"""


class ErrorExpresion(Exception):
    """Error de sintaxis o de evaluación en una expresión."""
    pass


def senalar(expr, pos, mensaje):
    """Arma un mensaje de error con una flecha (^) bajo la posición del problema."""
    return f"{mensaje}\n      {expr}\n      {' ' * pos}^"


def simbolo(valor):
    """Representación legible de un operador ('neg' se muestra como '-')."""
    return '-' if valor == 'neg' else valor

## Paso 4 · Módulo `lexico.py` (análisis léxico)

**Normalizar** unifica la escritura: `**` → `^`, `√` → `sqrt`, `×` → `*`, `[ ]` y `{ }` → `( )`, coma decimal → punto, mayúsculas → minúsculas.

**Tokenizar** separa la cadena en números, variables, constantes, funciones, operadores y paréntesis.

Validaciones: expresión vacía, signo `=`, caracteres no permitidos, números mal formados e identificadores desconocidos.

In [ ]:
%%writefile arbol_expresion/lexico.py
"""
lexico.py — Análisis léxico: normalización y tokenización.

Validaciones de esta etapa:
  - texto vacío o que no es str
  - uso del signo '='
  - caracteres no permitidos
  - números mal formados (3.4.5, '.')
  - identificadores desconocidos (raiz, sen, xy...)
"""
import re
from dataclasses import dataclass

from .config import FUNCIONES, CONSTANTES
from .errores import ErrorExpresion, senalar


@dataclass
class Token:
    tipo: str    # NUM, VAR, CONST, FUNC, OP, PAR_IZQ, PAR_DER
    valor: str
    pos: int     # posición en la cadena, para señalar errores


# Reemplazos para aceptar distintas formas de escribir lo mismo
_REEMPLAZOS = [
    ('**', '^'), ('×', '*'), ('·', '*'), ('÷', '/'), ('−', '-'),
    ('√', ' sqrt '), ('[', '('), (']', ')'), ('{', '('), ('}', ')'),
    (',', '.'),   # coma decimal -> punto decimal
]


def normalizar(expr):
    """Unifica la escritura de la expresión y valida que no esté vacía."""
    if not isinstance(expr, str):
        raise ErrorExpresion("La expresión debe ser un texto (str).")
    expr = expr.strip()
    if not expr:
        raise ErrorExpresion("La expresión está vacía.")
    if '=' in expr:
        raise ErrorExpresion("Escribe solo una expresión, sin el signo '=' "
                             "(por ejemplo: 2*x^2+5*x-3 en lugar de y=2*x^2+5*x-3).")
    for viejo, nuevo in _REEMPLAZOS:
        expr = expr.replace(viejo, nuevo)
    return re.sub(r'\s+', ' ', expr).strip().lower()


def _leer_numero(expr, i):
    texto = re.match(r'\d*\.?\d*', expr[i:]).group()
    fin = i + len(texto)
    if texto == '.':
        raise ErrorExpresion(senalar(expr, i, "Punto decimal sin dígitos."))
    if fin < len(expr) and expr[fin] == '.':
        raise ErrorExpresion(senalar(expr, fin, "Número mal formado (más de un punto decimal)."))
    return Token('NUM', texto, i), fin


def _leer_identificador(expr, i):
    palabra = re.match(r'[a-z]+', expr[i:]).group()
    if palabra in FUNCIONES:
        tipo = 'FUNC'
    elif palabra in CONSTANTES:
        tipo = 'CONST'
    elif len(palabra) == 1:
        tipo = 'VAR'
    else:
        raise ErrorExpresion(senalar(
            expr, i,
            f"Identificador desconocido '{palabra}'.\n"
            f"      Funciones válidas: {', '.join(FUNCIONES)}. Constantes: {', '.join(CONSTANTES)}.\n"
            f"      Las variables son de una letra; para multiplicar variables escribe x*y."))
    return Token(tipo, palabra, i), i + len(palabra)


_SIMBOLOS = {'+': 'OP', '-': 'OP', '*': 'OP', '/': 'OP', '^': 'OP',
             '(': 'PAR_IZQ', ')': 'PAR_DER'}


def tokenizar(expr):
    """Convierte la cadena normalizada en una lista de Token."""
    tokens, i = [], 0
    while i < len(expr):
        c = expr[i]
        if c.isspace():
            i += 1
        elif c.isdigit() or c == '.':
            tok, i = _leer_numero(expr, i)
            tokens.append(tok)
        elif c.isascii() and c.isalpha():
            tok, i = _leer_identificador(expr, i)
            tokens.append(tok)
        elif c in _SIMBOLOS:
            tokens.append(Token(_SIMBOLOS[c], c, i))
            i += 1
        else:
            raise ErrorExpresion(senalar(expr, i, f"Carácter no válido '{c}'."))
    return tokens

## Paso 5 · Módulo `sintaxis.py` (preprocesamiento y validación)

- Convierte el `-` unario en el operador `neg` (el `+` unario se descarta).
- Inserta **multiplicaciones implícitas**: `2x` → `2*x`, `3(x+1)` → `3*(x+1)`, `(x+1)(x-1)` → `(x+1)*(x-1)`.
- Valida con una **máquina de estados** que alterna entre *"espero un operando"* y *"espero un operador"*. Detecta operadores seguidos, inicio o fin con operador, paréntesis vacíos o desbalanceados, funciones sin paréntesis y números pegados.

In [ ]:
%%writefile arbol_expresion/sintaxis.py
"""
sintaxis.py — Preprocesamiento y validación sintáctica.

preprocesar():
  - convierte el '-' unario en el operador 'neg' (y descarta el '+' unario)
  - inserta multiplicaciones implícitas: 2x, 3(x+1), (x+1)(x-1), 2sqrt(x)

validar_sintaxis(): máquina de estados que alterna entre
"espero un operando" y "espero un operador".
"""
from .config import TIPOS_OPERANDO
from .errores import ErrorExpresion, senalar, simbolo
from .lexico import Token


def preprocesar(tokens, expr):
    """Regresa (tokens_procesados, avisos)."""
    resultado, avisos = [], []
    for tok in tokens:
        prev = resultado[-1] if resultado else None

        # 1) Multiplicación implícita
        if prev is not None:
            termina_operando = prev.tipo in TIPOS_OPERANDO + ('PAR_DER',)
            inicia_operando = tok.tipo in TIPOS_OPERANDO + ('FUNC', 'PAR_IZQ')
            if termina_operando and inicia_operando:
                if prev.tipo == 'NUM' and tok.tipo == 'NUM':
                    raise ErrorExpresion(senalar(
                        expr, tok.pos, f"Falta un operador entre {prev.valor} y {tok.valor}."))
                resultado.append(Token('OP', '*', tok.pos))
                avisos.append(f"Multiplicación implícita insertada: {prev.valor} * {tok.valor}")
                prev = resultado[-1]

        # 2) Signo unario: al inicio, después de un operador o después de '('
        if tok.tipo == 'OP' and tok.valor in ('+', '-'):
            if prev is None or prev.tipo in ('OP', 'PAR_IZQ'):
                if tok.valor == '+':
                    continue
                tok = Token('OP', 'neg', tok.pos)

        resultado.append(tok)
    return resultado, avisos


def _error_esperando_operando(tok, anterior, siguiente, expr):
    """Valida un token cuando se esperaba un operando. Regresa True si ya llegó el operando."""
    if tok.tipo in TIPOS_OPERANDO:
        return True
    if tok.tipo == 'PAR_IZQ' or (tok.tipo == 'OP' and tok.valor == 'neg'):
        return False
    if tok.tipo == 'FUNC':
        if siguiente is None or siguiente.tipo != 'PAR_IZQ':
            raise ErrorExpresion(senalar(
                expr, tok.pos,
                f"La función '{tok.valor}' debe ir seguida de '('. Ejemplo: {tok.valor}(x)"))
        return False
    if tok.tipo == 'PAR_DER':
        msg = ("Paréntesis vacíos '()'." if anterior is not None and anterior.tipo == 'PAR_IZQ'
               else "Falta un operando antes de ')'.")
        raise ErrorExpresion(senalar(expr, tok.pos, msg))
    # Operador binario donde se esperaba un operando
    if anterior is None:
        msg = f"La expresión no puede iniciar con el operador '{tok.valor}'."
    elif anterior.tipo == 'OP':
        msg = f"Dos operadores seguidos: '{simbolo(anterior.valor)}' y '{tok.valor}'."
    else:
        msg = f"Falta un operando antes de '{tok.valor}'."
    raise ErrorExpresion(senalar(expr, tok.pos, msg))


def validar_sintaxis(tokens, expr):
    """Lanza ErrorExpresion si la secuencia de tokens no es una expresión válida."""
    if not tokens:
        raise ErrorExpresion("La expresión no contiene elementos válidos.")

    espera_operando = True
    abiertos = []                      # posiciones de '(' sin cerrar

    for i, tok in enumerate(tokens):
        anterior = tokens[i - 1] if i > 0 else None
        siguiente = tokens[i + 1] if i + 1 < len(tokens) else None

        if espera_operando:
            if tok.tipo == 'PAR_IZQ':
                abiertos.append(tok.pos)
            if _error_esperando_operando(tok, anterior, siguiente, expr):
                espera_operando = False
        elif tok.tipo == 'OP':
            espera_operando = True
        elif tok.tipo == 'PAR_DER':
            if not abiertos:
                raise ErrorExpresion(senalar(expr, tok.pos, "Paréntesis ')' sin su '(' correspondiente."))
            abiertos.pop()
        else:
            raise ErrorExpresion(senalar(expr, tok.pos, f"Falta un operador antes de '{tok.valor}'."))

    if espera_operando:
        ultimo = tokens[-1]
        raise ErrorExpresion(senalar(
            expr, ultimo.pos, f"La expresión está incompleta: termina en '{simbolo(ultimo.valor)}'."))
    if abiertos:
        n = len(abiertos)
        raise ErrorExpresion(senalar(
            expr, abiertos[-1], f"Falta{'n' if n > 1 else ''} {n} paréntesis de cierre ')'."))

## Paso 6 · Módulo `postfija.py` (Shunting-Yard)

Convierte la notación **infija** a **postfija** y guarda cada paso del algoritmo en una tabla (token, acción, pila, salida).

In [ ]:
%%writefile arbol_expresion/postfija.py
"""
postfija.py — Conversión infija -> postfija con el algoritmo Shunting-Yard (Dijkstra).

Reglas:
  - Operando           -> salida
  - Función o '('      -> pila
  - Operador binario   -> antes de apilarlo salen los de MAYOR precedencia
                          (o IGUAL si es asociativo por la izquierda)
  - Operador unario    -> se apila directamente (es prefijo)
  - ')'                -> desapilar hasta '('; si debajo hay función, también sale
  - Fin                -> vaciar la pila
"""
from .config import OPERADORES, TIPOS_OPERANDO
from .errores import ErrorExpresion


def _texto(lista):
    return ' '.join(t.valor for t in lista) if lista else '∅'


def a_postfija(tokens):
    """Regresa (postfija, pasos). 'pasos' es la tabla del algoritmo para mostrarla."""
    salida, pila, pasos = [], [], []

    def anotar(token, accion):
        pasos.append((token, accion, _texto(pila), _texto(salida)))

    for tok in tokens:
        if tok.tipo in TIPOS_OPERANDO:
            salida.append(tok)
            anotar(tok.valor, "Operando → salida")

        elif tok.tipo in ('FUNC', 'PAR_IZQ'):
            pila.append(tok)
            anotar(tok.valor, "Función → pila" if tok.tipo == 'FUNC' else "'(' → pila")

        elif tok.tipo == 'PAR_DER':
            while pila and pila[-1].tipo != 'PAR_IZQ':
                salida.append(pila.pop())
            if not pila:
                raise ErrorExpresion("Paréntesis desbalanceados.")
            pila.pop()                                    # descarta '('
            accion = "Desapilar hasta '('"
            if pila and pila[-1].tipo == 'FUNC':
                salida.append(pila.pop())
                accion += " y sacar la función"
            anotar(')', accion)

        elif tok.tipo == 'OP':
            prec, asoc, aridad = OPERADORES[tok.valor]
            sacados = 0
            if aridad == 2:
                while pila and pila[-1].tipo == 'OP':
                    prec_tope = OPERADORES[pila[-1].valor][0]
                    if prec_tope > prec or (prec_tope == prec and asoc == 'izq'):
                        salida.append(pila.pop())
                        sacados += 1
                    else:
                        break
            pila.append(tok)
            anotar(tok.valor, "Operador → pila" + (f" (antes salen {sacados})" if sacados else ""))

    while pila:
        tope = pila.pop()
        if tope.tipo == 'PAR_IZQ':
            raise ErrorExpresion("Paréntesis desbalanceados.")
        salida.append(tope)
    anotar('fin', "Vaciar la pila → salida")
    return salida, pasos


def imprimir_pasos(pasos):
    """Imprime la tabla de Shunting-Yard."""
    ancho = max(len("Pila"), *(len(p[2]) for p in pasos))
    encabezado = f"{'#':>3} │ {'Token':<6} │ {'Acción':<40} │ {'Pila':<{ancho}} │ Salida"
    print(encabezado)
    print('─' * (len(encabezado) + 20))
    for k, (tok, accion, pila, salida) in enumerate(pasos, 1):
        print(f"{k:>3} │ {tok:<6} │ {accion:<40} │ {pila:<{ancho}} │ {salida}")

## Paso 7 · Módulo `arbol.py` (estructura, construcción y recorridos)

| Recorrido | Orden de visita | Notación que produce |
|---|---|---|
| **Preorden** | raíz → izquierdo → derecho | prefija (polaca) |
| **Inorden** | izquierdo → raíz → derecho | infija (sin paréntesis) |
| **Postorden** | izquierdo → derecho → raíz | postfija (polaca inversa) |

Las funciones y el menos unario tienen un solo hijo, que se coloca como hijo **derecho**.

In [ ]:
%%writefile arbol_expresion/arbol.py
"""
arbol.py — Estructura del árbol de expresión, construcción y recorridos.

Construcción a partir de la postfija con una pila:
  - Operando         -> hoja, se apila
  - Operador binario -> se desapilan dos nodos (primero el DERECHO, luego el IZQUIERDO)
  - Función / neg    -> se desapila un nodo que queda como hijo DERECHO
Al final la pila debe tener exactamente un nodo: la raíz.

Recorridos:
  Preorden  = raíz, izquierdo, derecho  -> notación prefija (polaca)
  Inorden   = izquierdo, raíz, derecho  -> notación infija (sin paréntesis)
  Postorden = izquierdo, derecho, raíz  -> notación postfija (polaca inversa)
"""
from .config import TIPOS_OPERANDO, OPERADORES
from .errores import ErrorExpresion


class Nodo:
    def __init__(self, valor, tipo, izq=None, der=None):
        self.valor = valor   # texto que se muestra en el nodo
        self.tipo = tipo     # NUM, VAR, CONST, OP (binario), OP_UN (neg), FUNC
        self.izq = izq
        self.der = der

    def es_hoja(self):
        return self.izq is None and self.der is None

    def __repr__(self):
        return f"Nodo({self.valor!r})"


def construir_arbol(postfija):
    """Construye el árbol a partir de una lista de tokens en postfija y regresa la raíz."""
    pila = []
    for tok in postfija:
        if tok.tipo in TIPOS_OPERANDO:
            pila.append(Nodo(tok.valor, tok.tipo))
        elif tok.tipo == 'FUNC' or tok.valor == 'neg':
            if not pila:
                raise ErrorExpresion(f"Falta el operando de '{tok.valor}'.")
            tipo = 'FUNC' if tok.tipo == 'FUNC' else 'OP_UN'
            pila.append(Nodo(tok.valor, tipo, der=pila.pop()))
        else:
            if len(pila) < 2:
                raise ErrorExpresion(f"Faltan operandos para '{tok.valor}'.")
            der, izq = pila.pop(), pila.pop()
            pila.append(Nodo(tok.valor, 'OP', izq, der))
    if len(pila) != 1:
        raise ErrorExpresion("Expresión mal formada: sobran o faltan operandos.")
    return pila[0]


# ---------- Recorridos ----------
def preorden(n):
    return [] if n is None else [n.valor] + preorden(n.izq) + preorden(n.der)


def inorden(n):
    return [] if n is None else inorden(n.izq) + [n.valor] + inorden(n.der)


def postorden(n):
    return [] if n is None else postorden(n.izq) + postorden(n.der) + [n.valor]


def infija_con_parentesis(n):
    """Reconstruye la expresión con todos los paréntesis (conserva el significado)."""
    if n.tipo in TIPOS_OPERANDO:
        return n.valor
    if n.tipo == 'FUNC':
        return f"{n.valor}({infija_con_parentesis(n.der)})"
    if n.tipo == 'OP_UN':
        return f"(-{infija_con_parentesis(n.der)})"
    return f"({infija_con_parentesis(n.izq)} {n.valor} {infija_con_parentesis(n.der)})"


def _precedencia(n):
    """Precedencia del nodo; hojas y funciones nunca necesitan paréntesis (5)."""
    if n.tipo in ('OP', 'OP_UN'):
        return OPERADORES[n.valor][0]
    return 5


def infija_minima(n):
    """Reconstruye la expresión solo con los paréntesis necesarios (la forma más legible)."""
    if n.tipo in TIPOS_OPERANDO:
        return n.valor
    if n.tipo == 'FUNC':
        return f"{n.valor}({infija_minima(n.der)})"
    if n.tipo == 'OP_UN':
        hijo = infija_minima(n.der)
        return f"-{hijo}" if _precedencia(n.der) >= 4 else f"-({hijo})"

    p = _precedencia(n)
    izq, der = infija_minima(n.izq), infija_minima(n.der)
    pi, pd = _precedencia(n.izq), _precedencia(n.der)
    if pi < p or (pi == p and n.valor == '^'):
        izq = f"({izq})"
    if pd < p or (pd == p and n.valor != '^') or der.startswith('-'):
        der = f"({der})"
    sep = ' ' if n.valor in ('+', '-') else ''
    return f"{izq}{sep}{n.valor}{sep}{der}"


# ---------- Estadísticas ----------
def altura(n):
    return 0 if n is None else 1 + max(altura(n.izq), altura(n.der))


def contar_nodos(n):
    return 0 if n is None else 1 + contar_nodos(n.izq) + contar_nodos(n.der)


def contar_hojas(n):
    if n is None:
        return 0
    return 1 if n.es_hoja() else contar_hojas(n.izq) + contar_hojas(n.der)


def variables(n):
    if n is None:
        return set()
    propia = {n.valor} if n.tipo == 'VAR' else set()
    return propia | variables(n.izq) | variables(n.der)

## Paso 8 · Módulo `evaluador.py` (evaluación con validación de dominio)

Recorre el árbol en postorden y detecta: raíz de negativo, logaritmo de un valor ≤ 0, tangente indefinida, división entre cero, potencias con resultado complejo y desbordamiento.

In [ ]:
%%writefile arbol_expresion/evaluador.py
"""
evaluador.py — Evaluación numérica del árbol (recorrido en postorden).

Validaciones de dominio:
  - raíz cuadrada de negativo
  - logaritmo de cero o negativo
  - tangente indefinida
  - división entre cero (incluye 0 elevado a negativo)
  - potencia de base negativa con exponente no entero (resultado complejo)
  - desbordamiento / resultados no finitos
"""
import math

from .config import FUNCIONES, CONSTANTES
from .errores import ErrorExpresion


def validar_valores(valores):
    """Revisa que 'valores' sea un dict de números finitos. Regresa una copia limpia."""
    if valores is None:
        return {}
    if not isinstance(valores, dict):
        raise ErrorExpresion("'valores' debe ser un diccionario, por ejemplo {'x': 2}.")
    limpios = {}
    for k, v in valores.items():
        if isinstance(v, bool) or not isinstance(v, (int, float)):
            raise ErrorExpresion(f"El valor de '{k}' debe ser numérico (recibido: {v!r}).")
        if not math.isfinite(v):
            raise ErrorExpresion(f"El valor de '{k}' debe ser un número finito.")
        limpios[str(k).lower()] = float(v)
    return limpios


def _aplicar_funcion(nombre, a):
    if nombre == 'sqrt' and a < 0:
        raise ErrorExpresion(f"Error de dominio: sqrt({a:g}) — raíz cuadrada de un número negativo.")
    if nombre in ('log', 'ln') and a <= 0:
        raise ErrorExpresion(f"Error de dominio: {nombre}({a:g}) — el logaritmo requiere un valor mayor que 0.")
    if nombre == 'tan' and abs(math.cos(a)) < 1e-12:
        raise ErrorExpresion(f"Error de dominio: tan({a:g}) no está definida.")
    try:
        return FUNCIONES[nombre](a)
    except OverflowError:
        raise ErrorExpresion(f"Desbordamiento al calcular {nombre}({a:g}).")


def _aplicar_operador(op, a, b):
    if op == '/' and b == 0:
        raise ErrorExpresion(f"División entre cero: {a:g} / 0.")
    if op == '^':
        if a == 0 and b < 0:
            raise ErrorExpresion(f"División entre cero: 0^{b:g}.")
        if a < 0 and not float(b).is_integer():
            raise ErrorExpresion(f"Error de dominio: ({a:g})^{b:g} da un número complejo.")
    operaciones = {
        '+': lambda: a + b,
        '-': lambda: a - b,
        '*': lambda: a * b,
        '/': lambda: a / b,
        '^': lambda: a ** b,
    }
    if op not in operaciones:
        raise ErrorExpresion(f"Operador desconocido '{op}'.")
    try:
        r = operaciones[op]()
    except OverflowError:
        raise ErrorExpresion(f"Desbordamiento al calcular {a:g} {op} {b:g}.")
    if not math.isfinite(r):
        raise ErrorExpresion(f"El resultado de {a:g} {op} {b:g} no es un número finito.")
    return r


def evaluar(n, valores=None):
    """Evalúa el árbol con los valores de las variables. Lanza ErrorExpresion si falla."""
    valores = valores or {}
    if n.tipo == 'NUM':
        return float(n.valor)
    if n.tipo == 'CONST':
        return CONSTANTES[n.valor]
    if n.tipo == 'VAR':
        if n.valor not in valores:
            raise ErrorExpresion(f"Falta el valor de la variable '{n.valor}'.")
        return float(valores[n.valor])
    if n.tipo == 'OP_UN':
        return -evaluar(n.der, valores)
    if n.tipo == 'FUNC':
        return _aplicar_funcion(n.valor, evaluar(n.der, valores))
    return _aplicar_operador(n.valor, evaluar(n.izq, valores), evaluar(n.der, valores))

## Paso 9 · Módulo `graficos.py` (dibujo del árbol y de la función)

- **x** de cada nodo = su posición en el recorrido inorden (no se enciman); **y** = su profundidad.
- Los nodos unarios se centran sobre su hijo. Colores por tipo de nodo y aristas marcadas con **I** (izquierdo) o **D** (derecho).
- Si hay una sola variable, se grafica `f(variable)` omitiendo los puntos fuera del dominio.

In [ ]:
%%writefile arbol_expresion/graficos.py
"""
graficos.py — Dibujo del árbol y gráfica de la función con matplotlib.

Distribución del árbol:
  x = posición del nodo en el recorrido inorden (así nunca se enciman)
  y = profundidad del nodo
Los nodos unarios (funciones y neg) se centran encima de su único hijo.
"""
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from .config import COLORES
from .arbol import altura, infija_minima
from .errores import ErrorExpresion
from .evaluador import evaluar


def _calcular_posiciones(raiz):
    posiciones, nodos, contador = {}, [], [0]

    def asignar(n, prof):
        if n is None:
            return
        asignar(n.izq, prof + 1)
        posiciones[id(n)] = (contador[0], -prof)
        nodos.append(n)
        contador[0] += 1
        asignar(n.der, prof + 1)

    def centrar_unarios(n):
        if n is None:
            return
        centrar_unarios(n.izq)
        centrar_unarios(n.der)
        if n.tipo in ('FUNC', 'OP_UN'):
            posiciones[id(n)] = (posiciones[id(n.der)][0], posiciones[id(n)][1])

    asignar(raiz, 0)
    centrar_unarios(raiz)
    return posiciones, nodos


def dibujar_arbol(raiz, titulo="Árbol de expresión", guardar_como=None):
    """Dibuja el árbol. Si se da 'guardar_como', también lo guarda como imagen."""
    posiciones, nodos = _calcular_posiciones(raiz)
    h = altura(raiz)
    n_nodos = len(nodos)
    ancho = min(max(7, 1.0 * n_nodos), 30)          # se limita para árboles grandes
    fuente = 12 if n_nodos <= 30 else (10 if n_nodos <= 60 else 8)
    fig, ax = plt.subplots(figsize=(ancho, max(4, 1.3 * h + 1)))

    # Aristas con la etiqueta I (izquierdo) o D (derecho)
    for n in nodos:
        x0, y0 = posiciones[id(n)]
        for hijo, lado in ((n.izq, 'I'), (n.der, 'D')):
            if hijo is not None:
                x1, y1 = posiciones[id(hijo)]
                ax.plot([x0, x1], [y0, y1], color='#555555', lw=1.6, zorder=1)
                ax.text((x0 + x1) / 2, (y0 + y1) / 2 + 0.08, lado, fontsize=8,
                        color='#888888', ha='center', va='center', zorder=2)

    # Nodos coloreados por tipo
    for n in nodos:
        x, y = posiciones[id(n)]
        forma = 'circle,pad=0.45' if len(n.valor) <= 2 else 'round,pad=0.45'
        ax.text(x, y, n.valor, ha='center', va='center', fontsize=fuente, fontweight='bold',
                color='white', zorder=3,
                bbox=dict(boxstyle=forma, fc=COLORES[n.tipo][0], ec='black', lw=1.2))

    presentes = {n.tipo for n in nodos}
    leyenda = [Patch(fc=color, ec='black', label=etiqueta)
               for tipo, (color, etiqueta) in COLORES.items() if tipo in presentes]
    ax.legend(handles=leyenda, loc='upper left', bbox_to_anchor=(1.01, 1), frameon=False)

    ax.set_xlim(-1, len(nodos))
    ax.set_ylim(-(h - 1) - 0.6, 0.6)
    ax.set_title(titulo, fontsize=13, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    if guardar_como:
        plt.savefig(guardar_como, dpi=200, bbox_inches='tight')
        print(f"💾 Árbol guardado como '{guardar_como}'")
    plt.show()


def graficar_funcion(raiz, var, rango=(-10, 10), puntos=1000):
    """Grafica f(var). Los puntos fuera del dominio se omiten."""
    try:
        a, b = rango
    except (TypeError, ValueError):
        print("⚠️ El rango debe ser una pareja (mínimo, máximo).")
        return
    if not all(isinstance(v, (int, float)) for v in (a, b)) or a >= b:
        print("⚠️ Rango inválido: debe ser (mínimo, máximo) con mínimo < máximo.")
        return

    xs = np.linspace(a, b, puntos)
    ys = []
    for x in xs:
        try:
            ys.append(evaluar(raiz, {var: x}))
        except ErrorExpresion:
            ys.append(np.nan)
    ys = np.array(ys, dtype=float)
    if np.all(np.isnan(ys)):
        print(f"⚠️ La función no está definida en ningún punto de [{a}, {b}].")
        return

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(xs, ys, lw=2, color='#4C72B0', label=f"f({var}) = {infija_minima(raiz)}")
    ax.axhline(0, color='black', lw=0.8)
    ax.axvline(0, color='black', lw=0.8)
    ax.grid(alpha=0.3)
    ax.set_xlabel(var)
    ax.set_ylabel(f"f({var})")
    ax.set_title("Gráfica de la función (se omiten los puntos fuera del dominio)")
    ax.legend(loc='best', fontsize=9)
    plt.tight_layout()
    plt.show()

## Paso 10 · Módulo `analizador.py` (integración)

**Clase `ArbolExpresion`:** construye el árbol al crearse (si la expresión no es válida lanza `ErrorExpresion`) y ofrece los métodos
`preorden()`, `inorden()`, `postorden()`, `infija()`, `variables()`, `estadisticas()`, `evaluar()`, `mostrar_pasos()`, `dibujar()` y `graficar()`.

**Función `analizar()`:** imprime el reporte completo y nunca detiene el programa por un error de entrada.

In [ ]:
%%writefile arbol_expresion/analizador.py
"""
analizador.py — Punto de entrada: une todas las etapas.

  texto --(lexico)--> tokens --(sintaxis)--> tokens válidos
        --(postfija)--> postfija --(arbol)--> árbol
        --(evaluador / graficos)--> resultados

Dos formas de uso:
  1) Orientada a objetos:  a = ArbolExpresion("sqrt(2*(x^2)+5*x-3)")
                           a.preorden(); a.evaluar(x=2); a.dibujar()
  2) Reporte completo:     analizar("sqrt(2*(x^2)+5*x-3)", valores={'x': 2})
"""
from .errores import ErrorExpresion
from .lexico import normalizar, tokenizar
from .sintaxis import preprocesar, validar_sintaxis
from .postfija import a_postfija, imprimir_pasos
from . import arbol as _arbol
from .evaluador import evaluar as _evaluar, validar_valores
from .graficos import dibujar_arbol, graficar_funcion


class ArbolExpresion:
    """Árbol de expresión construido a partir de un texto. Lanza ErrorExpresion si el texto no es válido.

    altura_maxima (opcional): si el árbol resultante es más alto, se rechaza la expresión.
    """

    def __init__(self, expresion, altura_maxima=None):
        if altura_maxima is not None and (isinstance(altura_maxima, bool)
                                          or not isinstance(altura_maxima, int) or altura_maxima < 1):
            raise ErrorExpresion("La altura máxima debe ser un entero mayor o igual a 1.")
        self.original = expresion
        self.normalizada = normalizar(expresion)
        tokens = tokenizar(self.normalizada)
        self.tokens, self.avisos = preprocesar(tokens, self.normalizada)
        validar_sintaxis(self.tokens, self.normalizada)
        self.postfija, self.pasos = a_postfija(self.tokens)
        self.raiz = _arbol.construir_arbol(self.postfija)
        h = _arbol.altura(self.raiz)
        if altura_maxima is not None and h > altura_maxima:
            raise ErrorExpresion(f"El árbol tiene altura {h} y supera la altura máxima permitida ({altura_maxima}).")

    # ----- Recorridos -----
    def preorden(self):
        return _arbol.preorden(self.raiz)

    def inorden(self):
        return _arbol.inorden(self.raiz)

    def postorden(self):
        return _arbol.postorden(self.raiz)

    def infija(self):
        return _arbol.infija_con_parentesis(self.raiz)

    def infija_minima(self):
        return _arbol.infija_minima(self.raiz)

    def altura(self):
        return _arbol.altura(self.raiz)

    # ----- Información -----
    def variables(self):
        return sorted(_arbol.variables(self.raiz))

    def estadisticas(self):
        nodos = _arbol.contar_nodos(self.raiz)
        hojas = _arbol.contar_hojas(self.raiz)
        return {'nodos': nodos, 'hojas': hojas, 'internos': nodos - hojas,
                'altura': _arbol.altura(self.raiz), 'raiz': self.raiz.valor}

    def es_consistente(self):
        """El postorden del árbol debe coincidir con la postfija de Shunting-Yard."""
        return self.postorden() == [t.valor for t in self.postfija]

    # ----- Evaluación y gráficos -----
    def evaluar(self, valores=None, **kwargs):
        """Ejemplos: a.evaluar(x=2)   o   a.evaluar({'x': 2, 'y': 1})"""
        datos = dict(valores or {})
        datos.update(kwargs)
        return _evaluar(self.raiz, validar_valores(datos))

    def mostrar_pasos(self):
        imprimir_pasos(self.pasos)

    def dibujar(self, guardar_como=None):
        dibujar_arbol(self.raiz, titulo=f"Árbol de expresión: {self.normalizada}",
                      guardar_como=guardar_como)

    def graficar(self, rango=(-10, 10)):
        vars_ = self.variables()
        if len(vars_) != 1:
            print(f"⚠️ Solo se grafica si hay exactamente una variable (hay {len(vars_)}).")
            return
        graficar_funcion(self.raiz, vars_[0], rango=rango)

    def __repr__(self):
        return f"ArbolExpresion({self.normalizada!r})"


def construir(expresion, altura_maxima=None):
    """Atajo: regresa un ArbolExpresion (lanza ErrorExpresion si no es válida)."""
    return ArbolExpresion(expresion, altura_maxima=altura_maxima)


def analizar(expr, valores=None, mostrar_pasos=True, graficar=True,
             graficar_f=True, rango=(-10, 10), guardar_como=None,
             altura_maxima=None, biblioteca=None, titulo="Expresión recibida"):
    """Imprime el reporte completo. Regresa el ArbolExpresion, o None si la expresión no es válida.

    altura_maxima: rechaza árboles más altos que este valor.
    biblioteca:    objeto Biblioteca; rechaza expresiones que usen elementos que no estén en ella.
    """
    linea = '═' * 70
    print(f"{linea}\n📥 {titulo}: {expr}\n{linea}")

    try:
        a = ArbolExpresion(expr, altura_maxima=altura_maxima)
        if biblioteca is not None:
            fuera = biblioteca.verificar(a.raiz)
            if fuera:
                raise ErrorExpresion("Usa elementos que no están en la biblioteca: " + ', '.join(fuera))
    except ErrorExpresion as err:
        print(f"❌ Expresión NO válida:\n   {err}")
        return None

    print(f"✅ Expresión válida\n   Normalizada: {a.normalizada}")
    for aviso in a.avisos:
        print(f"   ℹ️  {aviso}")

    print("\n🔤 TOKENS\n   " + '  '.join(f"[{t.valor}]" for t in a.tokens))

    if mostrar_pasos:
        print("\n🧮 CONVERSIÓN INFIJA → POSTFIJA (Shunting-Yard)\n")
        a.mostrar_pasos()

    print("\n🌳 RECORRIDOS DEL ÁRBOL")
    print(f"   Preorden  (raíz-izq-der, prefija) : {' '.join(a.preorden())}")
    print(f"   Inorden   (izq-raíz-der, infija)  : {' '.join(a.inorden())}")
    print(f"   Postorden (izq-der-raíz, postfija): {' '.join(a.postorden())}")
    print(f"   Infija con paréntesis             : {a.infija()}")
    print(f"   Infija mínima (legible)           : {a.infija_minima()}")
    print(f"\n🔎 Verificación: postorden == postfija de Shunting-Yard → "
          f"{'✔ coinciden' if a.es_consistente() else '✘ NO coinciden'}")

    e = a.estadisticas()
    vars_expr = a.variables()
    print("\n📊 ESTADÍSTICAS")
    print(f"   Nodos: {e['nodos']} | Hojas: {e['hojas']} | Nodos internos: {e['internos']} | "
          f"Altura: {e['altura']}{f' (máx. {altura_maxima})' if altura_maxima else ''} | Raíz: '{e['raiz']}'")
    print(f"   Variables: {', '.join(vars_expr) if vars_expr else '(ninguna)'}")

    try:
        limpios = validar_valores(valores)
        sobrantes = set(limpios) - set(vars_expr)
        if sobrantes:
            print(f"   ⚠️ Se ignoran valores de variables que no aparecen: {', '.join(sorted(sobrantes))}")
        if not vars_expr or limpios:
            resultado = a.evaluar(limpios)
            asignacion = ', '.join(f"{k} = {limpios[k]:g}" for k in vars_expr)
            print(f"\n🧾 EVALUACIÓN{' con ' + asignacion if asignacion else ''}: {resultado:.10g}")
    except ErrorExpresion as err:
        print(f"\n⚠️ No se pudo evaluar: {err}")

    if graficar:
        a.dibujar(guardar_como=guardar_como)
    if graficar_f and len(vars_expr) == 1:
        a.graficar(rango=rango)
    return a

## Paso 11 · Módulo `generador.py` (biblioteca, valores válidos, altura máxima)

Se basa en los conceptos de **programación genética**:

| Concepto | Qué es | Ejemplo |
|---|---|---|
| **No terminales** | nodos internos: operadores y funciones | `+ - * / ^`, `neg`, `sqrt`, `sin` |
| **Terminales** | hojas: variables, constantes y **valores válidos** | `x`, `pi`, `1, 2, 3, 5` |
| **Altura máxima** | límite del tamaño del árbol (la raíz sola mide 1) | con altura 4 caben como máximo 2⁴ − 1 = 15 nodos |

**Métodos de generación:**
- **grow:** en cada nivel se elige al azar entre una hoja o un operador; produce árboles de forma variada.
- **full:** se colocan operadores hasta el último nivel; produce árboles "llenos" de altura exacta.
- **mixto** (*ramped half-and-half*): sortea la altura de cada árbol y alterna entre grow y full; produce la mayor diversidad.

**Garantías:** cada ecuación generada se convierte a texto y se vuelve a analizar con el mismo parser. Se verifica que el árbol resultante sea idéntico, que no supere la altura máxima, que use solo elementos de la biblioteca y, si se piden `puntos_validos`, que pueda evaluarse en esos puntos sin errores de dominio.

In [ ]:
%%writefile arbol_expresion/generador.py
"""
generador.py — Generación aleatoria de ecuaciones a partir de una biblioteca.

Conceptos (los mismos que en programación genética):
  - Biblioteca de NO TERMINALES (nodos internos): operadores binarios (+ - * / ^),
    el menos unario (neg) y funciones (sqrt, sin, ...).
  - Biblioteca de TERMINALES (hojas): variables, constantes y la lista de VALORES VÁLIDOS.
  - ALTURA MÁXIMA: limita el tamaño del árbol (la raíz sola tiene altura 1).

Métodos de generación:
  - 'grow'  : en cada nivel se elige al azar entre terminal y no terminal → árboles de forma variada.
  - 'full'  : se usan no terminales hasta el último nivel → árboles "llenos" de altura exacta.
  - 'mixto' : (ramped half-and-half) para cada árbol se sortea una altura entre 2 y la máxima,
              y se alterna entre 'grow' y 'full' → la mayor diversidad.

Cada árbol generado se convierte a texto, se vuelve a analizar con el mismo parser que usan
las expresiones escritas a mano y se verifica que el árbol obtenido sea idéntico al generado.
"""
import math
import random

from .config import OPERADORES, FUNCIONES, CONSTANTES, ALTURA_MAXIMA_LIMITE
from .errores import ErrorExpresion
from .arbol import Nodo, altura, contar_nodos, contar_hojas, infija_minima, variables
from .evaluador import evaluar
from .analizador import ArbolExpresion, analizar

METODOS = ('grow', 'full', 'mixto')


# ───────────────────────── Utilidades de validación ─────────────────────────
def _como_lista(nombre, valor):
    """Acepta lista, tupla o un solo elemento; rechaza tipos que no tengan sentido."""
    if valor is None:
        return []
    if isinstance(valor, (str, int, float)) and not isinstance(valor, bool):
        valor = [valor]
    if not isinstance(valor, (list, tuple, set)):
        raise ErrorExpresion(f"'{nombre}' debe ser una lista, por ejemplo ['+', '*'].")
    lista = list(valor)
    repetidos = sorted({str(v) for v in lista if lista.count(v) > 1})
    if repetidos:
        raise ErrorExpresion(f"'{nombre}' tiene elementos repetidos: {', '.join(repetidos)}.")
    return lista


def _texto_numero(v):
    """Escribe un número sin notación científica (el lexer no la acepta): 3.0 -> '3', 0.5 -> '0.5'."""
    if float(v).is_integer():
        return str(int(v))
    return f"{v:.10f}".rstrip('0').rstrip('.')


# ───────────────────────────────── Biblioteca ─────────────────────────────────
class Biblioteca:
    """Conjunto de elementos permitidos para construir ecuaciones.

    operadores : subconjunto de ['+', '-', '*', '/', '^', 'neg']
    funciones  : subconjunto de las funciones de config.FUNCIONES
    variables  : letras, por ejemplo ['x'] o ['x', 'y']
    valores    : lista de valores válidos (números ≥ 0) que pueden aparecer como hojas
    constantes : subconjunto de ['pi', 'e']
    """

    def __init__(self, operadores=('+', '-', '*', '/'), funciones=('sqrt',),
                 variables=('x',), valores=(1, 2, 3, 5), constantes=()):
        self.operadores = self._validar_operadores(_como_lista('operadores', operadores))
        self.funciones = self._validar_funciones(_como_lista('funciones', funciones))
        self.variables = self._validar_variables(_como_lista('variables', variables))
        self.valores = self._validar_valores(_como_lista('valores', valores))
        self.constantes = self._validar_constantes(_como_lista('constantes', constantes))

        # (valor, tipo de nodo, aridad)
        self.no_terminales = (
            [(op, 'OP', 2) for op in self.operadores if op != 'neg'] +
            [('neg', 'OP_UN', 1)] * ('neg' in self.operadores) +
            [(f, 'FUNC', 1) for f in self.funciones]
        )
        # (texto, tipo de nodo)
        self.terminales = (
            [(v, 'VAR') for v in self.variables] +
            [(_texto_numero(v), 'NUM') for v in self.valores] +
            [(c, 'CONST') for c in self.constantes]
        )
        if not self.terminales:
            raise ErrorExpresion("La biblioteca necesita al menos un terminal (variable, valor o constante).")
        if not self.no_terminales:
            raise ErrorExpresion("La biblioteca necesita al menos un operador o una función para formar ecuaciones.")

    # ----- validaciones -----
    @staticmethod
    def _validar_operadores(ops):
        for op in ops:
            if op not in OPERADORES:
                raise ErrorExpresion(f"Operador no válido '{op}'. Opciones: {', '.join(OPERADORES)}.")
        return ops

    @staticmethod
    def _validar_funciones(funcs):
        for f in funcs:
            if f not in FUNCIONES:
                raise ErrorExpresion(f"Función no válida '{f}'. Opciones: {', '.join(FUNCIONES)}.")
        return funcs

    @staticmethod
    def _validar_variables(vars_):
        for v in vars_:
            if not (isinstance(v, str) and len(v) == 1 and v.isascii() and v.isalpha() and v.islower()):
                raise ErrorExpresion(f"Variable no válida {v!r}: debe ser una sola letra minúscula (a-z).")
            if v in CONSTANTES:
                raise ErrorExpresion(f"'{v}' es una constante; no puede usarse como variable.")
        return vars_

    @staticmethod
    def _validar_valores(vals):
        limpios = []
        for v in vals:
            if isinstance(v, bool) or not isinstance(v, (int, float)):
                raise ErrorExpresion(f"Valor no válido {v!r}: los valores deben ser números.")
            if not math.isfinite(v):
                raise ErrorExpresion(f"Valor no válido {v}: debe ser un número finito.")
            if v < 0:
                raise ErrorExpresion(f"Valor no válido {v}: usa solo valores ≥ 0. "
                                     f"Para obtener negativos agrega el operador 'neg' a la biblioteca.")
            if v > 1e6:
                raise ErrorExpresion(f"Valor no válido {v}: el máximo permitido es 1,000,000.")
            limpios.append(float(v))
        return limpios

    @staticmethod
    def _validar_constantes(consts):
        for c in consts:
            if c not in CONSTANTES:
                raise ErrorExpresion(f"Constante no válida '{c}'. Opciones: {', '.join(CONSTANTES)}.")
        return consts

    # ----- uso -----
    def verificar(self, raiz):
        """Regresa la lista de elementos del árbol que NO pertenecen a la biblioteca (vacía si todo es válido)."""
        fuera = []

        def visitar(n):
            if n is None:
                return
            permitido = {
                'OP':    lambda: n.valor in self.operadores,
                'OP_UN': lambda: 'neg' in self.operadores,
                'FUNC':  lambda: n.valor in self.funciones,
                'VAR':   lambda: n.valor in self.variables,
                'NUM':   lambda: float(n.valor) in self.valores,
                'CONST': lambda: n.valor in self.constantes,
            }[n.tipo]()
            if not permitido:
                etiqueta = '-' if n.tipo == 'OP_UN' else n.valor
                if etiqueta not in fuera:
                    fuera.append(etiqueta)
            visitar(n.izq)
            visitar(n.der)

        visitar(raiz)
        return fuera

    def describir(self):
        print("📚 BIBLIOTECA")
        print(f"   No terminales → operadores: {', '.join(self.operadores) or '(ninguno)'}"
              f" | funciones: {', '.join(self.funciones) or '(ninguna)'}")
        print(f"   Terminales    → variables: {', '.join(self.variables) or '(ninguna)'}"
              f" | valores válidos: {', '.join(_texto_numero(v) for v in self.valores) or '(ninguno)'}"
              f" | constantes: {', '.join(self.constantes) or '(ninguna)'}")

    def __repr__(self):
        return (f"Biblioteca(operadores={self.operadores}, funciones={self.funciones}, "
                f"variables={self.variables}, valores={[_texto_numero(v) for v in self.valores]}, "
                f"constantes={self.constantes})")


# ─────────────────────────── Generador de ecuaciones ───────────────────────────
class GeneradorExpresiones:
    """Genera ecuaciones aleatorias que respetan la biblioteca y la altura máxima.

    altura_maxima       : 1 a ALTURA_MAXIMA_LIMITE (la raíz sola tiene altura 1)
    altura_minima       : descarta árboles más bajos (evita ecuaciones triviales como x + 2)
    metodo             : 'grow', 'full' o 'mixto'
    prob_terminal       : en 'grow', probabilidad de poner una hoja antes del último nivel
    semilla             : entero para obtener siempre las mismas ecuaciones (reproducible)
    exigir_variable     : la ecuación debe contener al menos una variable
    exponente_constante : en '^' el exponente es un valor pequeño de la lista (evita x^(x^x))
    puntos_validos      : dict {variable: [valores]}; la ecuación debe poder evaluarse en todos ellos
    max_intentos        : intentos antes de rendirse si no se cumplen las condiciones
    """

    def __init__(self, biblioteca, altura_maxima=4, altura_minima=1, metodo='grow', prob_terminal=0.3,
                 semilla=None, exigir_variable=True, exponente_constante=True,
                 puntos_validos=None, max_intentos=500):
        if not isinstance(biblioteca, Biblioteca):
            raise ErrorExpresion("'biblioteca' debe ser un objeto Biblioteca.")
        if isinstance(altura_maxima, bool) or not isinstance(altura_maxima, int):
            raise ErrorExpresion("La altura máxima debe ser un número entero.")
        if not 1 <= altura_maxima <= ALTURA_MAXIMA_LIMITE:
            raise ErrorExpresion(f"La altura máxima debe estar entre 1 y {ALTURA_MAXIMA_LIMITE}.")
        if isinstance(altura_minima, bool) or not isinstance(altura_minima, int):
            raise ErrorExpresion("La altura mínima debe ser un número entero.")
        if not 1 <= altura_minima <= altura_maxima:
            raise ErrorExpresion(f"La altura mínima debe estar entre 1 y la altura máxima ({altura_maxima}).")
        if metodo not in METODOS:
            raise ErrorExpresion(f"Método no válido '{metodo}'. Opciones: {', '.join(METODOS)}.")
        if isinstance(prob_terminal, bool) or not isinstance(prob_terminal, (int, float)) \
                or not 0 <= prob_terminal <= 1:
            raise ErrorExpresion("'prob_terminal' debe ser un número entre 0 y 1.")
        if semilla is not None and (isinstance(semilla, bool) or not isinstance(semilla, int)):
            raise ErrorExpresion("La semilla debe ser un número entero (o None).")
        if isinstance(max_intentos, bool) or not isinstance(max_intentos, int) or max_intentos < 1:
            raise ErrorExpresion("'max_intentos' debe ser un entero mayor o igual a 1.")
        if exigir_variable and not biblioteca.variables:
            raise ErrorExpresion("Se pidió exigir una variable, pero la biblioteca no tiene variables.")

        self.biblioteca = biblioteca
        self.altura_maxima = altura_maxima
        self.altura_minima = altura_minima
        self.metodo = metodo
        self.prob_terminal = float(prob_terminal)
        self.semilla = semilla
        self.exigir_variable = exigir_variable
        self.exponente_constante = exponente_constante
        self.puntos = self._validar_puntos(puntos_validos)
        self.max_intentos = max_intentos
        self._rng = random.Random(semilla)
        self._contador = 0

    def _validar_puntos(self, puntos):
        """Convierte {'x': [1, 2], 'y': [3, 4]} en [{'x': 1, 'y': 3}, {'x': 2, 'y': 4}]."""
        if puntos is None:
            return []
        if not isinstance(puntos, dict) or not puntos:
            raise ErrorExpresion("'puntos_validos' debe ser un diccionario, por ejemplo {'x': [1, 2, 3]}.")
        longitudes, normalizados = set(), {}
        for var, lista in puntos.items():
            if var not in self.biblioteca.variables:
                raise ErrorExpresion(f"La variable '{var}' de los puntos no está en la biblioteca.")
            if isinstance(lista, (int, float)) and not isinstance(lista, bool):
                lista = [lista]
            if not isinstance(lista, (list, tuple)):
                raise ErrorExpresion(f"Los puntos de '{var}' deben ser una lista de números.")
            lista = list(lista)
            normalizados[var] = lista
            if not lista:
                raise ErrorExpresion(f"La lista de puntos de '{var}' está vacía.")
            for v in lista:
                if isinstance(v, bool) or not isinstance(v, (int, float)) or not math.isfinite(v):
                    raise ErrorExpresion(f"Punto no válido para '{var}': {v!r}.")
            longitudes.add(len(lista))
        if len(longitudes) > 1:
            raise ErrorExpresion("Todas las listas de 'puntos_validos' deben tener la misma longitud.")
        n = longitudes.pop()
        return [{var: float(lista[i]) for var, lista in normalizados.items()} for i in range(n)]

    # ----- construcción aleatoria del árbol -----
    def _terminal(self):
        texto, tipo = self._rng.choice(self.biblioteca.terminales)
        return Nodo(texto, tipo)

    def _exponente(self):
        """Exponente de '^': prefiere enteros 2, 3 o 4 de la lista de valores válidos
        (evita potencias triviales como x^1 o x^0 y potencias que se desbordan)."""
        utiles = [v for v in self.biblioteca.valores if v in (2, 3, 4)]
        pequenos = utiles or [v for v in self.biblioteca.valores if 0 <= v <= 4]
        if pequenos:
            return Nodo(_texto_numero(self._rng.choice(pequenos)), 'NUM')
        return self._terminal()

    def _crecer(self, profundidad, h_max, metodo):
        """Crea un subárbol cuya raíz está en 'profundidad' (la raíz del árbol es profundidad 1)."""
        if profundidad == h_max:                                   # último nivel: solo hojas
            return self._terminal()
        es_raiz = profundidad == 1
        if metodo == 'grow' and not es_raiz and self._rng.random() < self.prob_terminal:
            return self._terminal()

        valor, tipo, aridad = self._rng.choice(self.biblioteca.no_terminales)
        if aridad == 1:
            return Nodo(valor, tipo, der=self._crecer(profundidad + 1, h_max, metodo))
        izq = self._crecer(profundidad + 1, h_max, metodo)
        if valor == '^' and self.exponente_constante:
            der = self._exponente()
        else:
            der = self._crecer(profundidad + 1, h_max, metodo)
        return Nodo(valor, tipo, izq, der)

    def _arbol_aleatorio(self):
        if self.altura_maxima == 1:
            return self._terminal(), 'grow', 1
        if self.metodo == 'mixto':
            h = self._rng.randint(max(2, self.altura_minima), self.altura_maxima)
            metodo = 'full' if self._contador % 2 else 'grow'
        else:
            h, metodo = self.altura_maxima, self.metodo
        return self._crecer(1, h, metodo), metodo, h

    def _cumple_dominio(self, raiz):
        for punto in self.puntos:
            try:
                evaluar(raiz, punto)
            except ErrorExpresion:
                return False
        return True

    # ----- interfaz pública -----
    def generar(self):
        """Genera una ecuación válida y la regresa como ArbolExpresion."""
        for intento in range(1, self.max_intentos + 1):
            raiz, metodo, h = self._arbol_aleatorio()
            if altura(raiz) < self.altura_minima:
                continue
            if self.exigir_variable and not variables(raiz):
                continue
            if not self._cumple_dominio(raiz):
                continue

            texto = infija_minima(raiz)
            arbol = ArbolExpresion(texto, altura_maxima=self.altura_maxima)
            if _firma(arbol.raiz) != _firma(raiz):           # verificación de ida y vuelta
                raise ErrorExpresion(f"Error interno: el texto '{texto}' no reproduce el árbol generado.")

            self._contador += 1
            arbol.metodo, arbol.intentos, arbol.altura_objetivo = metodo, intento, h
            return arbol
        raise ErrorExpresion(
            f"No se pudo generar una ecuación que cumpla las condiciones en {self.max_intentos} intentos. "
            f"Prueba con otra altura, otra biblioteca o con menos restricciones (puntos_validos).")

    def generar_lote(self, cantidad=10, mostrar=True, evaluar_en=None):
        """Genera varias ecuaciones. 'evaluar_en' = {'x': [0, 1, 2]} agrega columnas con f(x)."""
        if isinstance(cantidad, bool) or not isinstance(cantidad, int) or not 1 <= cantidad <= 500:
            raise ErrorExpresion("La cantidad debe ser un entero entre 1 y 500.")
        puntos = self._validar_puntos(evaluar_en) if evaluar_en else []
        lote = [self.generar() for _ in range(cantidad)]
        if mostrar:
            self._imprimir_lote(lote, puntos)
        return lote

    def _imprimir_lote(self, lote, puntos):
        def f_en(a, p):
            try:
                return f"{a.evaluar(p):.4g}"
            except ErrorExpresion:
                return "indef."

        cab_puntos = [', '.join(f"{k}={v:g}" for k, v in p.items()) for p in puntos]
        ancho_ec = max(len("Ecuación"), *(len(a.infija_minima()) for a in lote))
        enc = f"{'#':>3} │ {'Ecuación':<{ancho_ec}} │ Altura │ Nodos │ Hojas │ Método"
        enc += ''.join(f" │ f({c})" for c in cab_puntos)
        print(enc)
        print('─' * len(enc))
        for i, a in enumerate(lote, 1):
            e = a.estadisticas()
            fila = (f"{i:>3} │ {a.infija_minima():<{ancho_ec}} │ {e['altura']:>6} │ {e['nodos']:>5} │ "
                    f"{e['hojas']:>5} │ {a.metodo:<6}")
            fila += ''.join(f" │ {f_en(a, p):>{len(c) + 3}}" for p, c in zip(puntos, cab_puntos))
            print(fila)

    def analizar_nueva(self, valores=None, **opciones):
        """Genera una ecuación y le aplica el análisis completo (árbol, recorridos, gráficas...)."""
        arbol = self.generar()
        print(f"🎲 Ecuación generada #{self._contador} (método {arbol.metodo}, "
              f"intento {arbol.intentos}, altura máxima {self.altura_maxima})")
        return analizar(arbol.infija_minima(), valores=valores, altura_maxima=self.altura_maxima,
                        biblioteca=self.biblioteca, titulo="Ecuación generada", **opciones)

    def describir(self):
        self.biblioteca.describir()
        print(f"⚙️  GENERADOR → altura: {self.altura_minima} a {self.altura_maxima} | método: {self.metodo} | "
              f"prob. terminal: {self.prob_terminal:g} | semilla: {self.semilla} | "
              f"exigir variable: {'sí' if self.exigir_variable else 'no'}")
        if self.puntos:
            print("   Debe poder evaluarse en: " +
                  '; '.join(', '.join(f"{k}={v:g}" for k, v in p.items()) for p in self.puntos))


def _firma(n):
    """Representación estructural del árbol para comparar dos árboles."""
    if n is None:
        return None
    valor = _texto_numero(float(n.valor)) if n.tipo == 'NUM' else n.valor
    return (valor, n.tipo, _firma(n.izq), _firma(n.der))

## Paso 12 · Módulo `interactivo.py`

In [ ]:
%%writefile arbol_expresion/interactivo.py
"""
interactivo.py — Modo interactivo por consola (funciona en Colab con input()).
"""
from .errores import ErrorExpresion
from .analizador import ArbolExpresion, analizar


def _pedir_valores(variables):
    """Pide el valor de cada variable. Regresa None si el usuario deja uno vacío."""
    valores = {}
    for v in variables:
        while True:
            texto = input(f"   Valor de {v} (Enter para no evaluar): ").strip().replace(',', '.')
            if texto == '':
                return None
            try:
                valores[v] = float(texto)
                break
            except ValueError:
                print("   ⚠️ Escribe un número válido, por ejemplo 2 o -3.5")
    return valores


def modo_interactivo(mostrar_pasos=True):
    print("Escribe una expresión (o 'salir' para terminar).")
    print("Ejemplos: sqrt(2*(x^2)+5*x-3) | (a+b)*(a-b) | 3x^2 - 2x + 1 | sin(x)/x\n")
    while True:
        expr = input("➤ Expresión: ").strip()
        if expr.lower() in ('salir', 'exit', 'q'):
            print("👋 Fin del modo interactivo.")
            break
        try:
            a = ArbolExpresion(expr)
        except ErrorExpresion as err:
            print(f"❌ {err}\n")
            continue
        valores = _pedir_valores(a.variables())
        analizar(expr, valores=valores, mostrar_pasos=mostrar_pasos)
        print()

## Paso 13 · Módulo `pruebas.py`

Prueba la validación sintáctica, la precedencia y asociatividad, los errores de dominio, la altura máxima, las configuraciones inválidas de la biblioteca y del generador, y 900 ecuaciones generadas.

In [ ]:
%%writefile arbol_expresion/pruebas.py
"""
pruebas.py — Pruebas automáticas de validación, recorridos y evaluación.
"""
import math

from .errores import ErrorExpresion
from .analizador import ArbolExpresion

# (expresión, ¿debe ser válida?)
CASOS_SINTAXIS = [
    ("sqrt(2*(x^2)+5*x-3)", True),
    ("-x^2 + 3*(x - 1)/2", True),
    ("2^3^2", True),
    ("sin(pi/2) + ln(e)", True),
    ("(x+1)(x-1)", True),
    ("√(x)", True),
    ("", False),
    ("2*(x+3", False),
    ("2*x+3)", False),
    ("2*/x", False),
    ("*x+1", False),
    ("x+", False),
    ("()", False),
    ("sqrt x", False),
    ("raiz(x)", False),
    ("2 3 + x", False),
    ("3.4.5 + x", False),
    ("x & 2", False),
    ("y = 2x + 1", False),
]

# (expresión, valores, resultado esperado)
CASOS_EVALUACION = [
    ("sqrt(2*(x^2)+5*x-3)", {'x': 2}, math.sqrt(15)),
    ("2x^2 + 5x - 3", {'x': 2}, 15.0),
    ("2^3^2", {}, 512.0),          # asociatividad derecha
    ("-x^2", {'x': 3}, -9.0),      # el menos unario va después de la potencia
    ("10 - 4 - 3", {}, 3.0),       # asociatividad izquierda
    ("sin(pi/2) + ln(e)", {}, 2.0),
]

# (expresión, valores) que deben producir error de dominio
CASOS_DOMINIO = [
    ("sqrt(2*(x^2)+5*x-3)", {'x': 0}),
    ("1/(x-2)", {'x': 2}),
    ("ln(x)", {'x': -1}),
    ("(-8)^(1/3)", {}),
    ("exp(x)", {'x': 1000}),
]


def ejecutar_pruebas(detalle=True):
    total = aciertos = 0

    print("1) Validación sintáctica")
    for expr, esperado in CASOS_SINTAXIS:
        try:
            ArbolExpresion(expr)
            valida, msg = True, ""
        except ErrorExpresion as err:
            valida, msg = False, str(err).split('\n')[0]
        ok = valida == esperado
        total, aciertos = total + 1, aciertos + ok
        if detalle:
            estado = "VÁLIDA  " if valida else "INVÁLIDA"
            print(f"   {'✔' if ok else '✘'} {estado} │ {expr!r:<24} │ {msg}")

    print("\n2) Evaluación y precedencia")
    for expr, vals, esperado in CASOS_EVALUACION:
        a = ArbolExpresion(expr)
        r = a.evaluar(vals)
        ok = math.isclose(r, esperado, rel_tol=1e-9) and a.es_consistente()
        total, aciertos = total + 1, aciertos + ok
        if detalle:
            print(f"   {'✔' if ok else '✘'} {expr} con {vals} = {r:.6g} (esperado {esperado:.6g})")

    print("\n3) Errores de dominio")
    for expr, vals in CASOS_DOMINIO:
        try:
            ArbolExpresion(expr).evaluar(vals)
            ok, msg = False, "no se detectó el error"
        except ErrorExpresion as err:
            ok, msg = True, str(err)
        total, aciertos = total + 1, aciertos + ok
        if detalle:
            print(f"   {'✔' if ok else '✘'} {expr} con {vals}: {msg}")

    print("\n4) Altura máxima en expresiones escritas a mano")
    for expr, h_max, esperado in [("sqrt(2*(x^2)+5*x-3)", 6, True),
                                  ("sqrt(2*(x^2)+5*x-3)", 4, False),
                                  ("x", 1, True)]:
        try:
            ArbolExpresion(expr, altura_maxima=h_max)
            valida, msg = True, "aceptada"
        except ErrorExpresion as err:
            valida, msg = False, str(err)
        ok = valida == esperado
        total, aciertos = total + 1, aciertos + ok
        if detalle:
            print(f"   {'✔' if ok else '✘'} {expr} con altura máxima {h_max}: {msg}")

    print("\n5) Biblioteca y generador: configuraciones inválidas")
    from .generador import Biblioteca, GeneradorExpresiones
    malas = [
        ("operador desconocido", lambda: Biblioteca(operadores=['+', '%'])),
        ("función desconocida", lambda: Biblioteca(funciones=['raiz'])),
        ("variable de 2 letras", lambda: Biblioteca(variables=['xy'])),
        ("variable 'e'", lambda: Biblioteca(variables=['e'])),
        ("valor negativo", lambda: Biblioteca(valores=[-2])),
        ("valor no numérico", lambda: Biblioteca(valores=['3'])),
        ("valores repetidos", lambda: Biblioteca(valores=[1, 1])),
        ("sin terminales", lambda: Biblioteca(variables=[], valores=[])),
        ("sin operadores ni funciones", lambda: Biblioteca(operadores=[], funciones=[])),
        ("altura 0", lambda: GeneradorExpresiones(Biblioteca(), altura_maxima=0)),
        ("altura mayor al límite", lambda: GeneradorExpresiones(Biblioteca(), altura_maxima=50)),
        ("altura no entera", lambda: GeneradorExpresiones(Biblioteca(), altura_maxima=3.5)),
        ("altura mínima > máxima", lambda: GeneradorExpresiones(Biblioteca(), altura_maxima=3, altura_minima=5)),
        ("método desconocido", lambda: GeneradorExpresiones(Biblioteca(), metodo='otro')),
        ("probabilidad fuera de [0,1]", lambda: GeneradorExpresiones(Biblioteca(), prob_terminal=2)),
        ("puntos con variable ajena", lambda: GeneradorExpresiones(Biblioteca(), puntos_validos={'y': [1]})),
    ]
    for nombre, crear in malas:
        try:
            crear()
            ok, msg = False, "no se detectó"
        except ErrorExpresion as err:
            ok, msg = True, str(err)
        total, aciertos = total + 1, aciertos + ok
        if detalle:
            print(f"   {'✔' if ok else '✘'} {nombre}: {msg}")

    print("\n6) Generación: 300 ecuaciones por método respetan altura (3 a 5), biblioteca y dominio")
    bib = Biblioteca(operadores=['+', '-', '*', '/', '^', 'neg'], funciones=['sqrt', 'sin'],
                     variables=['x'], valores=[1, 2, 3, 0.5], constantes=['pi'])
    for metodo in ('grow', 'full', 'mixto'):
        gen = GeneradorExpresiones(bib, altura_maxima=5, altura_minima=3, metodo=metodo, semilla=7,
                                   puntos_validos={'x': [1, 2, 3]})
        problemas = 0
        for _ in range(300):
            a = gen.generar()
            if (not 3 <= a.altura() <= 5 or bib.verificar(a.raiz) or not a.variables()
                    or not a.es_consistente()):
                problemas += 1
            for x in (1, 2, 3):
                a.evaluar(x=x)            # no debe lanzar error
        ok = problemas == 0
        total, aciertos = total + 1, aciertos + ok
        if detalle:
            print(f"   {'✔' if ok else '✘'} método {metodo}: {problemas} ecuaciones con problemas")

    gen_a = GeneradorExpresiones(bib, altura_maxima=4, semilla=123)
    gen_b = GeneradorExpresiones(bib, altura_maxima=4, semilla=123)
    ok = [gen_a.generar().infija_minima() for _ in range(5)] == \
         [gen_b.generar().infija_minima() for _ in range(5)]
    total, aciertos = total + 1, aciertos + ok
    if detalle:
        print(f"   {'✔' if ok else '✘'} la misma semilla produce las mismas ecuaciones")

    print(f"\nResultado: {aciertos}/{total} pruebas correctas.")
    return aciertos, total

## Paso 14 · Módulo `__init__.py`

Define qué se puede importar directamente con `import arbol_expresion`.

In [ ]:
%%writefile arbol_expresion/__init__.py
"""
arbol_expresion — Paquete para construir, recorrer, evaluar y graficar árboles de expresión.

Módulos:
  config       Funciones, constantes, operadores y colores
  errores      Excepción ErrorExpresion y mensajes con señalamiento
  lexico       Normalización y tokenización
  sintaxis     Menos unario, multiplicación implícita y validación
  postfija     Algoritmo Shunting-Yard (infija -> postfija)
  arbol        Nodo, construcción, recorridos y estadísticas
  evaluador    Evaluación numérica con validación de dominio
  graficos     Dibujo del árbol y gráfica de la función
  analizador   Clase ArbolExpresion y función analizar()
  generador    Biblioteca, valores válidos, altura máxima y generación aleatoria de ecuaciones
  interactivo  Modo interactivo con input()
  pruebas      Pruebas automáticas
"""
from .errores import ErrorExpresion
from .analizador import ArbolExpresion, analizar, construir
from .generador import Biblioteca, GeneradorExpresiones
from .interactivo import modo_interactivo
from .pruebas import ejecutar_pruebas

__all__ = ['ErrorExpresion', 'ArbolExpresion', 'analizar', 'construir',
           'Biblioteca', 'GeneradorExpresiones', 'modo_interactivo', 'ejecutar_pruebas']
__version__ = '3.0'

## Paso 15 · Importar el paquete

`autoreload` hace que, si modificas un módulo y vuelves a ejecutar su celda `%%writefile`, los cambios se carguen sin reiniciar el entorno.

In [ ]:
%load_ext autoreload
%autoreload 2

import arbol_expresion as ae
from arbol_expresion import (ArbolExpresion, ErrorExpresion, analizar,
                             Biblioteca, GeneradorExpresiones)

print(f"✅ Paquete arbol_expresion v{ae.__version__} importado.")
print("   Archivos:", ', '.join(sorted(f for f in os.listdir('arbol_expresion') if f.endswith('.py'))))

## Paso 16 · Reporte completo con la ecuación de la imagen

$\sqrt{2x^2 + 5x - 3}$ evaluada en $x = 2$: $\sqrt{8 + 10 - 3} = \sqrt{15} \approx 3.873$

👉 **Para usar otra expresión, cambia `EXPRESION` y `VALORES` y vuelve a ejecutar esta celda.**

In [ ]:
EXPRESION = "sqrt(2*(x^2)+5*x-3)"
VALORES = {'x': 2}

arbol = analizar(EXPRESION, valores=VALORES, rango=(-8, 6), guardar_como="arbol_expresion.png")

### La segunda línea de la imagen, escrita "a mano"

`2x^2 + 5x - 3` usa multiplicación implícita; el programa la detecta y la convierte en `2*x^2 + 5*x - 3`.

In [ ]:
arbol2 = analizar("2x^2 + 5x - 3", valores={'x': 2}, mostrar_pasos=False, rango=(-5, 3))

## Paso 17 · Uso orientado a objetos con `ArbolExpresion`

Útil cuando solo necesitas una parte del resultado (por ejemplo, un recorrido o un valor).

In [ ]:
try:
    a = ArbolExpresion("sqrt(2*(x^2)+5*x-3)")
    print("Preorden :", ' '.join(a.preorden()))
    print("Inorden  :", ' '.join(a.inorden()))
    print("Postorden:", ' '.join(a.postorden()))
    print("Infija   :", a.infija())
    print("Variables:", a.variables())
    print("Estadísticas:", a.estadisticas())
    for x in [0.5, 1, 2, 5]:
        print(f"f({x}) = {a.evaluar(x=x):.6f}")
    a.evaluar(x=0)          # provoca un error de dominio a propósito
except ErrorExpresion as err:
    print("⚠️", err)

# 🎲 Generación automática de ecuaciones

## Paso 18 · Definir la biblioteca, los valores válidos y la altura máxima

👉 **Esta es la celda que debes modificar para tus pruebas.** Cambia los elementos permitidos, los valores válidos o la altura y vuelve a ejecutar las celdas siguientes.

| Parámetro | Para qué sirve |
|---|---|
| `operadores` | operadores permitidos: `'+', '-', '*', '/', '^', 'neg'` |
| `funciones` | funciones permitidas: `'sqrt', 'sin', 'cos', 'tan', 'log', 'ln', 'exp', 'abs'` |
| `variables` | variables (una letra cada una) |
| `valores` | **lista de valores válidos** que pueden aparecer como números en las hojas (≥ 0) |
| `constantes` | `'pi'` y/o `'e'` |
| `altura_maxima` | **altura máxima** del árbol (1 a 8) |
| `altura_minima` | altura mínima, para evitar ecuaciones triviales como `x + 2` |
| `metodo` | `'grow'`, `'full'` o `'mixto'` |
| `semilla` | número fijo para obtener siempre las mismas ecuaciones; `None` para que cambien cada vez |
| `puntos_validos` | la ecuación generada debe poder evaluarse en estos puntos (sin raíces de negativos, divisiones entre cero, etc.) |

In [ ]:
biblioteca = Biblioteca(
    operadores=['+', '-', '*', '/', '^'],
    funciones=['sqrt'],
    variables=['x'],
    valores=[1, 2, 3, 5],          # lista de valores válidos
    constantes=[],
)

generador = GeneradorExpresiones(
    biblioteca,
    altura_maxima=4,               # límite del tamaño del árbol
    altura_minima=3,
    metodo='mixto',
    semilla=2026,                  # None = ecuaciones distintas en cada ejecución
    puntos_validos={'x': [1, 2, 3]},
)

generador.describir()

## Paso 19 · Generar una ecuación y analizarla completa

Genera una ecuación nueva y le aplica todo el proceso: tabla de Shunting-Yard, recorridos, árbol, gráfica y evaluación.
Además verifica que respete la altura máxima y la biblioteca. **Ejecuta la celda varias veces para obtener ecuaciones distintas.**

In [ ]:
ecuacion = generador.analizar_nueva(valores={'x': 2}, rango=(0, 6), guardar_como="arbol_generado.png")

## Paso 20 · Generar un lote de ecuaciones

Genera varias ecuaciones y muestra una tabla con su altura, número de nodos y su valor en algunos puntos.

In [ ]:
lote = generador.generar_lote(10, evaluar_en={'x': [1, 2, 3]})

### Analizar en detalle una ecuación del lote

Cambia el número para ver el árbol de otra ecuación de la tabla.

In [ ]:
NUMERO = 1
analizar(lote[NUMERO - 1].infija_minima(), valores={'x': 2}, mostrar_pasos=False,
         altura_maxima=generador.altura_maxima, biblioteca=biblioteca, rango=(0, 6))

## Paso 21 · Comparar los tres métodos de generación

Misma biblioteca y misma altura máxima: `full` llena todos los niveles, `grow` produce formas variadas y `mixto` combina ambos.

In [ ]:
for metodo in ['full', 'grow', 'mixto']:
    print(f"\n▶ Método {metodo}")
    GeneradorExpresiones(biblioteca, altura_maxima=4, metodo=metodo, semilla=1,
                         puntos_validos={'x': [1, 2, 3]}).generar_lote(5)

## Paso 22 · La biblioteca y la altura máxima también validan tus ecuaciones

Con `altura_maxima` y `biblioteca`, `analizar()` rechaza expresiones demasiado grandes o que usen elementos no permitidos.

In [ ]:
# La ecuación de la imagen tiene altura 6: con altura máxima 4 se rechaza
analizar("sqrt(2*(x^2)+5*x-3)", altura_maxima=4, graficar=False)

# Con una biblioteca que solo permite + y *, también se rechaza
analizar("sqrt(2*(x^2)+5*x-3)", biblioteca=Biblioteca(operadores=['+', '*'], valores=[2, 3, 5]),
         graficar=False)

# Configuraciones inválidas de la biblioteca y del generador
for prueba in [lambda: Biblioteca(valores=[-2]),
               lambda: Biblioteca(operadores=['%']),
               lambda: GeneradorExpresiones(biblioteca, altura_maxima=20)]:
    try:
        prueba()
    except ErrorExpresion as err:
        print("⚠️", err)

## Paso 23 · Pruebas automáticas

In [ ]:
ae.ejecutar_pruebas()

## Paso 24 · Modo interactivo (opcional)

Escribe expresiones una por una. Si la expresión tiene variables, se piden sus valores (deja vacío para no evaluar). Escribe **`salir`** para terminar.

In [ ]:
ae.modo_interactivo()